In [1]:
import os
import re
import json
import argparse
from pathlib import Path
from typing import List, Dict

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm


ASPECT_KEYS = [
    "chu_the_hanh_dong",   # Subject & Action
    "vat_the_dac_diem",    # Objects & Visual Attributes
    "boi_canh_khong_gian", # Scene & Environment
    "chu_logo_man_hinh",   # Text, Logos, Screen
    "goc_nhin_co_canh",    # Camera angle, shot size, camera motion
    "dien_bien_thoi_gian", # Temporal progression
    "tong_the_canh_quay",  # Overall summary
    "full_caption",        # Complete raw caption text
]


NUM_GPUS = torch.cuda.device_count()

if NUM_GPUS > 0:
    print(f"🚀 CUDA available: {NUM_GPUS} GPU(s)")
    for i in range(NUM_GPUS):
        print(f"   GPU {i}: {torch.cuda.get_device_name(i)}")

    DEVICE = torch.device("cuda:0")
else:
    print("⚠️ CUDA not available. Using CPU.")
    DEVICE = torch.device("cpu")


def last_token_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]


def sanitize_caption_text(text: str, max_words: int = 2024) -> str:
    """
    Removes degenerate LLM repetition loops (repeated phrases or lines),
    fixes trailing unclosed quotes, removes truncated lines, and clamps words to max_words.
    """
    if not text or not isinstance(text, str):
        return text or ""

    # 1. Line deduplication FIRST (preserves newlines & collapses multi-line cycles)
    lines = text.split("\n")
    seen_lines = set()
    dedup_lines = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            dedup_lines.append(line)
            continue
        norm = re.sub(r"\s+", " ", stripped)
        if norm not in seen_lines:
            seen_lines.add(norm)
            dedup_lines.append(line)
    text = "\n".join(dedup_lines)

    # 2. Fast O(N) comma/semicolon phrase deduplication (avoids regex backtracking)
    if "," in text or ";" in text:
        delim = "," if text.count(",") >= text.count(";") else ";"
        lines = text.split("\n")
        cleaned_lines = []
        for line in lines:
            if delim not in line:
                cleaned_lines.append(line)
                continue
            parts = [p.strip() for p in line.split(delim)]
            seen_parts = set()
            dedup_parts = []
            for p in parts:
                norm_p = p.strip('\" ')
                if norm_p and norm_p in seen_parts:
                    continue
                if norm_p:
                    seen_parts.add(norm_p)
                dedup_parts.append(p)
            cleaned_lines.append((delim + " ").join(dedup_parts))
        text = "\n".join(cleaned_lines)

    # 4. Clean trailing unclosed quote or dangling quotation fragment
    text = text.strip()
    if text.count('\"') % 2 != 0:
        last_q = text.rfind('\"')
        prefix = text[:last_q].rstrip()
        if prefix.endswith((",", ";", "-", ":", "\n")):
            text = prefix.rstrip(",;-:\n ")
        else:
            text = text + '\"'

    # 5. Clean trailing cut-off line (incomplete sentence without terminating punctuation)
    lines = text.split("\n")
    if len(lines) > 1:
        last_line = lines[-1].strip()
        valid_endings = (".", "!", "?", '"', "”", "’", ")", "]", "}", ":")
        if last_line and not last_line.endswith(valid_endings):
            if last_line.startswith(("-", "*", "•")) or len(last_line.split()) < 8:
                lines.pop()
                text = "\n".join(lines)

    # 6. Clamp to max_words (default: 2024 words)
    words = text.split()
    if len(words) > max_words:
        text = " ".join(words[:max_words])

    return text.strip()


class Qwen3EmbeddingWrapper(nn.Module):
    """
    Runs Qwen3 encoder and performs last-token pooling INSIDE
    each DataParallel replica.

    This prevents DataParallel from gathering the full
    [batch, seq_len, hidden_dim] tensor onto GPU 0.
    """

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):

        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        last_hidden_states = outputs.last_hidden_state

        # Last-token pooling
        left_padding = (
            attention_mask[:, -1].sum()
            == attention_mask.shape[0]
        )

        if left_padding:

            pooled = last_hidden_states[:, -1]

        else:

            sequence_lengths = (
                attention_mask.sum(dim=1) - 1
            )

            batch_size = last_hidden_states.shape[0]

            pooled = last_hidden_states[
                torch.arange(
                    batch_size,
                    device=last_hidden_states.device,
                ),
                sequence_lengths,
            ]

        # Normalize BEFORE DataParallel gathers
        pooled = F.normalize(
            pooled,
            p=2,
            dim=1,
        )

        return pooled


class Qwen3CaptionEmbedder:
    def __init__(
        self,
        model_id: str = "Qwen/Qwen3-Embedding-0.6B",
        device: str = None,
        max_length: int = 2048,
        batch_size: int = 128,
    ):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.max_length = max_length
        self.batch_size = batch_size

        print(f"Loading tokenizer & model '{model_id}' on {self.device}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")
        
        # Use float16 on GPU (optimal for Kaggle T4/P100), float32 on CPU
        torch_dtype = torch.float16 if self.device == "cuda" else torch.float32
        
        # Try SDPA (Scaled Dot Product Attention) for memory-efficient attention
        try:
            self.model = AutoModel.from_pretrained(
                model_id,
                dtype=torch_dtype,
                attn_implementation="sdpa",
            ).to(self.device)
        except Exception:
            self.model = AutoModel.from_pretrained(
                model_id,
                dtype=torch_dtype,
            ).to(self.device)

        self.model.eval()
        print("Model loaded successfully.")


class Qwen3CaptionEmbedder:

    def __init__(
        self,
        model_id: str = "Qwen/Qwen3-Embedding-0.6B",
        max_length: int = 2048,
        batch_size: int = 128,
    ):

        self.max_length = max_length
        self.batch_size = batch_size

        self.use_cuda = torch.cuda.is_available()

        if self.use_cuda:
            self.device = torch.device("cuda:0")
        else:
            self.device = torch.device("cpu")

        print()
        print("=" * 70)
        print("Loading Qwen3 Embedding")
        print("=" * 70)
        print(f"Model       : {model_id}")
        print(f"Device      : {self.device}")
        print(f"Batch size  : {batch_size}")
        print(f"Max length  : {max_length}")
        print(f"GPUs        : {NUM_GPUS}")

        # ----------------------------------------------------
        # Tokenizer
        # ----------------------------------------------------

        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            padding_side="left",
        )

        # ----------------------------------------------------
        # Model
        # ----------------------------------------------------

        if self.use_cuda:

            torch_dtype = torch.float16

            try:
                model = AutoModel.from_pretrained(
                    model_id,
                    dtype=torch_dtype,
                    attn_implementation="sdpa",
                )

            except Exception:
                print(
                    "⚠️ SDPA unavailable, "
                    "falling back to default attention."
                )

                model = AutoModel.from_pretrained(
                    model_id,
                    dtype=torch_dtype,
                )

            # IMPORTANT:
            # Move base model to GPU 0 first.
            model = model.to(self.device)

            # ------------------------------------------------
            # Multi-GPU DataParallel
            # ------------------------------------------------

            if NUM_GPUS >= 2:

                print(
                    f"🔥 Using DataParallel on GPUs "
                    f"0-{NUM_GPUS - 1}"
                )

                # Pool inside each GPU replica before gather
                model = Qwen3EmbeddingWrapper(model)
                
                self.model = nn.DataParallel(
                    model,
                    device_ids=list(range(NUM_GPUS)),
                    output_device=0,
                )
                
            else:

                print("Using single GPU.")

                self.model = model

        else:

            self.model = AutoModel.from_pretrained(
                model_id,
                dtype=torch.float32,
            )

            self.model = self.model.to(self.device)

        self.model.eval()

        print("✅ Model loaded successfully.")
        print("=" * 70)
        print()
    
    @torch.no_grad()
    def _encode_batch(self, batch_texts: List[str]) -> torch.Tensor:
        """Encodes a batch of texts and returns L2-normalized float32 tensor (B, D) on CPU."""
        # Sanitize all texts in the batch to avoid runaway LLM loops or OOM spikes
        clean_batch = [sanitize_caption_text(t, max_words=self.max_length) for t in batch_texts]

        batch_dict = self.tokenizer(
            clean_batch,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        # Move input to GPU 0.
        #
        # DataParallel will automatically scatter
        # the batch across GPU 0 and GPU 1.
        batch_dict = {
            k: v.to(
                self.device,
                non_blocking=True,
            )
            for k, v in batch_dict.items()
        }

        # DataParallel:
        #
        # GPU 0:
        #   ~batch/2 -> Qwen -> pooling -> embeddings
        #
        # GPU 1:
        #   ~batch/2 -> Qwen -> pooling -> embeddings
        #
        # Only embeddings are gathered to GPU 0.
        embs = self.model(**batch_dict)
    
        return embs.cpu().float()

    
    def process_video_caption(
        self,
        caption_json_path: Path,
        output_dir: Path,
        overwrite: bool = False
    ) -> bool:
        video_id = caption_json_path.stem
        out_file = output_dir / f"{video_id}.pt"

        if out_file.exists() and not overwrite:
            tqdm.write(f"⏩ [{video_id}] Already embedded ({out_file.name}). Skipping.")
            return False

        with open(caption_json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        shots = data.get("shots", [])
        num_shots = len(shots)
        if num_shots == 0:
            tqdm.write(f"⚠️ [{video_id}] 0 shots found. Skipping.")
            return False

        # Gather metadata
        shot_indices = [s["shot_idx"] for s in shots]
        start_frames = [s["start_frame"] for s in shots]
        end_frames = [s["end_frame"] for s in shots]
        rep_frames = [
            s.get("sampled_keyframes", [s["start_frame"]])[len(s.get("sampled_keyframes", [])) // 2]
            for s in shots
        ]

        # Collect all sentences across all shots and all aspects for this video
        all_texts = []
        for s in shots:
            for aspect in ASPECT_KEYS:
                if aspect == "full_caption":
                    txt = s.get("caption_text", "").strip()
                else:
                    txt = s.get("aspects", {}).get(aspect, "").strip()
                raw_txt = txt if txt else s.get("caption_text", "không có")
                cleaned_txt = sanitize_caption_text(raw_txt, max_words=self.max_length)
                all_texts.append(cleaned_txt if cleaned_txt else "không có")

        total_sentences = len(all_texts)
        num_aspects = len(ASPECT_KEYS)
        all_embeddings_list = []

        # Track progress by NUMBER OF SENTENCES (unit="sent")
        with tqdm(total=total_sentences, desc=f"🎬 [{video_id}] ({num_shots} shots)", unit="sent", leave=False) as sent_pbar:
            for i in range(0, total_sentences, self.batch_size):
                batch_texts = all_texts[i : i + self.batch_size]
                embs = self._encode_batch(batch_texts)
                all_embeddings_list.append(embs)
                sent_pbar.update(len(batch_texts))

        # Concatenate and reshape: (num_shots * 8, emb_dim) -> (num_shots, 8, emb_dim)
        emb_dim = all_embeddings_list[0].shape[-1]
        all_embs_tensor = torch.cat(all_embeddings_list, dim=0).view(num_shots, num_aspects, emb_dim)

        final_aspect_embeddings = {
            aspect: all_embs_tensor[:, a_idx, :].contiguous()
            for a_idx, aspect in enumerate(ASPECT_KEYS)
        }

        # Save dictionary
        payload = {
            "video_id": video_id,
            "total_shots": num_shots,
            "shot_indices": shot_indices,
            "start_frames": start_frames,
            "end_frames": end_frames,
            "representative_frames": rep_frames,
            "embeddings": final_aspect_embeddings, # Dict[aspect_name -> Tensor (N, emb_dim)]
        }

        output_dir.mkdir(parents=True, exist_ok=True)
        torch.save(payload, out_file)
        tqdm.write(f"✅ [{video_id}] {total_sentences}/{total_sentences} sentences encoded ({num_shots} shots) -> saved to '{out_file.name}'.")
        return True

🚀 CUDA available: 2 GPU(s)
   GPU 0: Tesla T4
   GPU 1: Tesla T4


In [2]:
caption_dir = Path("/kaggle/input/datasets/ntloi131205/aic2026-shot-captioning/caption")
output_dir = Path("./embedding")
video_id=None
batch_size=256
model_id="Qwen/Qwen3-Embedding-0.6B"
prefix=None
overwrite=False
max_length=2048

embedder = Qwen3CaptionEmbedder(
    model_id=model_id,
    batch_size=batch_size,
)


Loading Qwen3 Embedding
Model       : Qwen/Qwen3-Embedding-0.6B
Device      : cuda:0
Batch size  : 256
Max length  : 2048
GPUs        : 2


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

🔥 Using DataParallel on GPUs 0-1
✅ Model loaded successfully.



In [3]:
# Filter files
if video_id:
    target_files = [caption_dir / f"{video_id}.json"]
elif prefix:
    target_files = sorted(caption_dir.glob(f"{prefix}_*.json"))
else:
    target_files = sorted(caption_dir.glob("*.json"))

target_files = [f for f in target_files if f.exists()]
total_shots_all = 0
for f in target_files:
    try:
        with open(f, "r", encoding="utf-8") as jf:
            total_shots_all += len(json.load(jf).get("shots", []))
    except Exception:
        pass
total_sent_all = total_shots_all * len(ASPECT_KEYS)
print(f"Found {len(target_files)} caption file(s) to process ({total_shots_all:,} shots = {total_sent_all:,} sentences).")

embedder = Qwen3CaptionEmbedder(
    model_id=model_id,
    batch_size=batch_size,
    max_length=max_length,
)

video_pbar = tqdm(target_files, desc="🎥 Total Videos", unit="vid")
for cfile in video_pbar:
    embedder.process_video_caption(cfile, output_dir, overwrite=overwrite)

print(f"\n🎉 All embeddings successfully saved in: {output_dir}")

Found 1 caption file(s) to process (330 shots = 2,640 sentences).

Loading Qwen3 Embedding
Model       : Qwen/Qwen3-Embedding-0.6B
Device      : cuda:0
Batch size  : 256
Max length  : 2048
GPUs        : 2


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

🔥 Using DataParallel on GPUs 0-1
✅ Model loaded successfully.



🎥 Total Videos: 100%|██████████| 1/1 [01:16<00:00, 76.93s/vid]                 

✅ [L21_V001] 2640/2640 sentences encoded (330 shots) -> saved to 'L21_V001.pt'.

🎉 All embeddings successfully saved in: embedding


In [4]:
!zip -r results.zip /kaggle/working
from IPython.display import FileLink
FileLink(r'results.zip')

  adding: kaggle/working/ (stored 0%)
  adding: kaggle/working/embedding/ (stored 0%)
  adding: kaggle/working/embedding/L21_V001.pt (deflated 42%)
  adding: kaggle/working/.virtual_documents/ (stored 0%)
  adding: kaggle/working/.virtual_documents/__notebook_source__.ipynb (deflated 69%)


/kaggle/working/results.zip